# S3 · Attention-pool construction and exact-identity validation

Pool homogeneity uses the exact identity `mean cosine = focal vector · mean pool vector`. The full implementation maintains sparse rolling vector sums; this notebook verifies its brute-force checks and empirical support.

In [1]:
from pathlib import Path
import json, time
import pandas as pd
import numpy as np
import psutil
from IPython.display import display, Image

ROOT = Path.cwd().parent
OUTPUTS = ROOT / "outputs"
AUDIT = ROOT / "audit"
LOGS = ROOT / "logs"
LOGS.mkdir(exist_ok=True)
RUN_LOG = LOGS / "run_log.txt"

def checkpoint(label, *frames, started=None):
    elapsed = time.time() - started if started is not None else 0.0
    shapes = [getattr(x, "shape", None) for x in frames]
    nulls = []
    for frame in frames:
        if hasattr(frame, "isna"):
            nulls.append(round(float(frame.isna().mean(numeric_only=False).mean()), 6))
    rss = psutil.Process().memory_info().rss / 1024**3
    line = f"{label} | shapes={shapes} | mean_null_rates={nulls} | rss_gib={rss:.3f} | elapsed_s={elapsed:.3f}"
    print(line)
    with RUN_LOG.open("a", encoding="utf-8") as stream:
        stream.write(line + "\n")

t0 = time.time()
print(f"Audit root: {ROOT}")
checkpoint("setup", started=t0)

Audit root: <PROJECT_ROOT>/5_最终交付包/_rebuild/MA-Hackathon-Final-2026-09-04
setup | shapes=[] | mean_null_rates=[] | rss_gib=0.113 | elapsed_s=0.000


In [2]:
t0 = time.time()
identity = pd.read_csv(OUTPUTS / "pool_identity_check.csv")
identity_summary = pd.read_csv(OUTPUTS / "pool_identity_summary.csv")
pool_dist = pd.read_csv(OUTPUTS / "pool_size_distribution.csv")
display(identity_summary)
display(pool_dist)
max_error = max(identity["abs_error_raw"].max(), identity["abs_error_residual"].max())
assert max_error <= 1e-9
print(f"PASS: {len(identity):,} brute-force comparisons; max abs error={max_error:.3e}")
checkpoint("S3 pool identity", identity, identity_summary, pool_dist, started=t0)

,focal_loans,pool_checks,max_abs_error_raw,max_abs_error_residual,acceptance_threshold,passed
0,200,600,2.908784e-14,2.509104e-14,1.000000e-09,True


,period,pool,loans,p01,p25,p50,p75,p99,pct_below_5,pct_below_10,pct_below_20
0,all,active,1453840.0,5.000000,246.000000,758.000000,1242.000000,3198.000000,0.991787,2.027871,4.169578
1,all,posting_14d,1453840.0,31.000000,412.000000,1129.000000,1639.000000,3049.000000,0.124567,0.245419,0.467108
2,all,posting_16d,1453840.0,36.000000,463.000000,1284.000000,1832.000000,3378.000000,0.107852,0.218043,0.384912
3,all,lag_kish,1453840.0,13.230686,483.653699,1450.411581,2036.069134,3663.493271,0.841702,0.955195,1.070063
4,all,lag_completed_kish,1453840.0,13.230686,483.083216,1438.557123,2010.248901,3647.640726,0.842734,0.955195,1.071370
5,train_2016_2024,active,1316678.0,5.000000,241.000000,759.000000,1262.000000,3233.000000,0.992042,2.023274,4.105256
6,train_2016_2024,posting_14d,1316678.0,32.000000,401.000000,1129.000000,1641.000000,3044.000000,0.126455,0.243340,0.433060
7,train_2016_2024,posting_16d,1316678.0,37.000000,451.000000,1285.000000,1838.000000,3413.000000,0.109518,0.219188,0.361972
8,train_2016_2024,lag_kish,1316678.0,12.996494,466.172995,1441.695299,2024.029594,3676.050208,0.849259,0.966523,1.082877
9,train_2016_2024,lag_completed_kish,1316678.0,12.995318,465.483949,1428.370604,1994.302516,3656.620516,0.850398,0.966523,1.084320


PASS: 600 brute-force comparisons; max abs error=2.909e-14
S3 pool identity | shapes=[(600, 9), (1, 6), (15, 11)] | mean_null_rates=[0.0, 0.0, 0.0] | rss_gib=0.114 | elapsed_s=0.010


In [3]:
t0 = time.time()
support = pd.read_csv(OUTPUTS / "joint_support_summary.csv")
cells = pd.read_csv(OUTPUTS / "joint_support_quartile_cells.csv")
manifest = json.loads((AUDIT / "joint_support_manifest.json").read_text())
display(support)
assert manifest["joint_support_pass"]
print("PASS: each P25/P75 endpoint has at least 1,000 nearby observed training loans")
checkpoint("S3 joint support", support, cells, started=t0)

,channel,endpoint,left_variable,right_variable,left_target,right_target,left_halfwidth_0_10_iqr,right_halfwidth_0_10_iqr,n_near_endpoint,share_main_train,median_funding_hours_near,fast_72h_rate_near,left_right_correlation,main_train_n
0,current,P25_P25,C_active,H_active_raw,5.641907,0.026199,0.152049,0.007906,12310,0.009975,39.530972,0.567506,-0.153179,1234131
1,current,P75_P75,C_active,H_active_raw,7.162397,0.105255,0.152049,0.007906,7953,0.006444,128.799167,0.398089,-0.153179,1234131
2,recent,P25_P25,V_lag,G_lag_raw,5.635039,0.026115,0.133243,0.007093,4716,0.003821,134.433472,0.412850,-0.228604,1234131
3,recent,P75_P75,V_lag,G_lag_raw,6.967471,0.097042,0.133243,0.007093,8168,0.006618,48.737083,0.542360,-0.228604,1234131


PASS: each P25/P75 endpoint has at least 1,000 nearby observed training loans
S3 joint support | shapes=[(4, 14), (32, 8)] | mean_null_rates=[0.0, 0.0] | rss_gib=0.114 | elapsed_s=0.004
